# Aula 2 — Entender os dados e modelar uma relação

**Disciplina 2 · Aprendizado de Máquina · IASEG**

Ontem: o pipeline inteiro, do 1 ao 7, com o Iris. Hoje a gente volta para a
**Etapa 2** e fica lá quase a manhã inteira.

| | Etapa | |
|---|---|---|
| 1 | Definição do problema | |
| **2** | **Preparação dos dados** | ← **hoje** |
| 3 | Divisão treino / teste | |
| 4 | Pré-processamento | |
| 5 | Treinamento do modelo | |
| 6 | Predição | |
| 7 | Avaliação | |

*Notebook de vocês.*

## Configuração

`pandas` para os dados, `matplotlib` para os gráficos. O resto é importado na
etapa em que for usado.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

## Etapa 1 — Definição do problema

O alvo natural desta base é `y`: a pessoa aceitou ou não. Isso é uma **categoria**,
ou seja, **classificação** — e não é o que a gente vai fazer hoje.

A pergunta de hoje:

> **Dá para prever a _idade_ de uma pessoa a partir do perfil dela?**

Saída é número → **regressão**. Cada linha tem o valor certo → **supervisionado**.

⭐ O interessante não vai ser a previsão. Vai ser **em que colunas o modelo se
apoiou** para chegar nela.

## Etapa 2 — Preparação dos dados

Uma linha, uma pessoa. O arquivo é separado por **ponto e vírgula**.

In [ ]:
URL = "https://raw.githubusercontent.com/HegdeChaitra/Bank-Marketing-Campaign-Analysis/master/bank-additional-full.csv"

df = pd.read_csv(URL, sep=';')

print('Linhas:', df.shape[0], '| Colunas:', df.shape[1])

In [ ]:
# Vamos exibir todas as colunas disponíveis no dataset
print('\nColunas disponíveis no dataset:')
for col in df.columns:
    print(col)

21 colunas, misturando texto e número.

In [ ]:
df.head()

### Estatísticas descritivas

`describe()` resume cada coluna **numérica**.

> ⭐ **Olhem essa tabela com calma. Alguma coisa aqui parece estranha?**

In [ ]:
df.describe()

### Histogramas

Onde os valores de cada coluna se concentram — e onde não tem nada.

Um deles não parece um histograma. Parece uma barra só.

In [ ]:
df[['age', 'campaign', 'pdays', 'previous']].hist(
    figsize=(10, 7), bins=30, edgecolor='black')
plt.suptitle('Distribuição de quatro colunas numéricas', fontsize=14)
plt.tight_layout()
plt.show()

---

# 🔍 A auditoria

**Em duplas. Todo mundo olha as mesmas cinco colunas:**
`education` · `default` · `pdays` · `poutcome` · `month`

```python
df['coluna'].value_counts()   # texto
df['coluna'].describe()       # número
```

As células abaixo já estão escritas. **O trabalho não é escrever código** — é olhar
o que sai e preencher a coluna da direita.

### ✍️ Pontos de atenção

Não é uma nota para a coluna. É o recado que vocês deixariam para a próxima pessoa
que for usar esses dados: **o que ela precisa saber antes de usar essa coluna?**

| coluna | o que significa | pontos de atenção |
|---|---|---|
| `education` | |  |
| `default` | | |
| `pdays` | | |
| `poutcome` | | |
| `month` | | |

⚡ **Terminaram antes?** *Quantas linhas vocês jogariam fora se decidissem não usar
essa coluna?*

In [ ]:
df['education'].value_counts()

In [ ]:
df['default'].value_counts()

In [ ]:
df['pdays'].describe()

In [ ]:
df['poutcome'].value_counts()

In [ ]:
df['month'].value_counts()

### ✍️ O que essas cinco células mostraram

-
-
-
-
-

---

## E o que o `pandas` diz que está faltando?

In [ ]:
# o que o pandas acha que está faltando
df.isna().sum()

O `pandas` diz (corretamente) que **não falta nada** nesta base. De fato não há nenhum dado faltante formalmente (não há nenhum "nan"/"na"). O que encontramos foram outros tipos de dados anômalos/inesperados.

---

⚠️ **Mas encontrar não é resolver.** Cada coluna dessas vira uma **decisão**: devemos escolher usar, descartar, ou tratar, por exemplo, o `unknown` como categoria de verdade.

Hoje a gente vai **tratar `unknown` como categoria**.

**✍️ Vocês concordam? Escrevam uma linha dizendo por quê, ou o que fariam
diferente.** Isso é a **seção 2** do relatório.

>

### E mais uma, que só aparece se você perguntar

`.duplicated()` é uma pergunta sobre a **tabela inteira**, não sobre uma coluna.

In [ ]:
# linhas repetidas na tabela inteira
print('Linhas repetidas:', df.duplicated().sum())

---

## A quinta propriedade: como o alvo se distribui

- Se for **categoria**: quantos de cada.
- Se for **número**: qual a faixa, e **onde a base é rala**.

In [ ]:
# como o alvo se distribui: y (categoria) e age (número)
print(df['y'].value_counts(normalize=True))
print('\n')
print(df['age'].describe())

Em barras, que é como isto aparece num relatório:

In [ ]:
# o mesmo em barras


**✍️ De cada cem pessoas para quem o banco ligou, quantas aceitaram?**

> resposta:

⭐ **Anotem esse número.** Ele vai importar amanhã mais do que vocês imaginam.

---

## A decisão: o que entra no modelo

Nem toda coluna precisa entrar. **Escolher é a parte que é de vocês**, e é o que
precisa estar escrito no relatório.

Hoje: **as seis colunas que descrevem a pessoa** — `job`, `marital`, `education`,
`default`, `housing`, `loan`.

- As colunas de economia (`euribor3m`, `nr.employed`…) descrevem **o país**.
- As de campanha (`campaign`, `pdays`, `previous`, `contact`…) descrevem **o que o
  banco fez**.

Podia ser diferente. Mas tem que estar escrito.

In [ ]:
# COLUNAS, X e y — são seis, e elas descrevem a PESSOA
# O alvo natural da base (categoria)
print(df['y'].value_counts(normalize=True).round(3))
print()
# O alvo de hoje (número)
print(df['age'].describe().round(1))
print()
print('Pessoas com mais de 70 anos:',
      f'{(df["age"] > 70).mean() * 100:.1f}% da base')

### Cada coluna de texto vira várias colunas de 0 e 1

> 🔎 **Por que isto está na Etapa 2 e não na 4?** Porque não estamos *aprendendo*
> nada com os dados aqui — só reescrevendo a tabela num formato que o modelo entende.
> Por isso pode ser feito antes da divisão treino/teste, sem risco de vazamento.

In [ ]:
# get_dummies
df['y'].value_counts().plot(kind='bar', color='#2166AC', rot=0)
plt.title('Quantas pessoas aceitaram o investimento')
plt.ylabel('número de pessoas')
plt.show()

## Etapa 3 — Divisão treino / teste

Igual a ontem. *(Sem `stratify` hoje: aquilo mantinha a proporção de categorias, e o
nosso alvo agora é um número.)*

In [ ]:
# train_test_split


## Etapa 4 — Pré-processamento

**Hoje não tem nada a fazer aqui.** Ontem a gente padronizou porque o k-NN mede
distâncias. A regressão linear não mede distância; padronizar não mudaria o resultado.

> ⭐ **A etapa continua na lista mesmo sem fazer nada.** O pipeline é uma
> **checagem**, não uma receita obrigatória.

## Etapa 5 — Treinamento

Uma linha para achar os parâmetros.

In [ ]:
# LinearRegression + fit


## Etapa 6 — Predição

In [ ]:
# predict, e comparar com a idade real


---

## Em que o modelo se apoiou?

Como o alvo é idade, **cada peso está em anos**: quanto aquela característica soma ou
tira da idade prevista.

> ### ⭐ Antes de rodar a próxima célula:
>
> **Das seis colunas que entraram, qual vocês acham que o modelo mais usou?**
>
> *(Anotem o palpite. É a mesma pergunta que eu fiz ontem, na tabela do empréstimo,
> e não respondi.)*

**✍️ Palpite da dupla:** ______________

In [ ]:
# os pesos, ordenados pelo tamanho


In [ ]:
# os mesmos pesos, em gráfico de barras


### ✍️ Lendo isso em voz alta

O maior peso da lista é ______________, com ______ anos.

**Isso faz sentido para vocês?**

> resposta:

E os dois seguintes: ______________ e ______________.

⭐ **Três apoios, não um.** Se fosse só o primeiro podia ser sorte.

Essa é a **pergunta 6** do banco de perguntas do projeto.

### E ainda falta um número

A lista acima tem **27 pesos**. Mas o modelo guarda **28** números — o que falta é o
**ponto de partida**, de onde a conta começa antes de somar qualquer coluna.

In [ ]:
# o ponto de partida do modelo


## Etapa 7 — Avaliação

Ontem, no Iris, a avaliação foi intuitiva: acertou a espécie ou não acertou. A gente
contou os acertos e dividiu pelo total — `accuracy_score`.

**Hoje não dá.** O modelo devolveu 50,5 anos para uma pessoa de 55. Isso é um
acerto? **Não existe "acertou" quando a resposta é um número** — o `accuracy_score`
nem roda aqui.

O que dá para medir é **o tamanho do erro**.

In [ ]:
# mean_absolute_error


**✍️ Qual foi o erro médio?** ______ anos

**Esse número é bom?** Essa pergunta tem outra na frente dela:

> **bom comparado com o quê?**

⭐ É a **aula 3** inteira. Guardem dois números: os **93,3%** do Iris de ontem, e
este de hoje.

---

# 📌 Tarefa — seções 1 e 2 do relatório

**Para a monitoria de hoje à tarde (14h, com o Lucas — obrigatória) e para amanhã.**

### 1. A frase do grupo *(seção 1)*

> *"Queremos prever ______ a partir de ______, e o modelo devolve ______
> (um número / uma categoria / uma probabilidade)."*

### 2. A auditoria da base de vocês *(seção 2)*

As cinco perguntas desta manhã, nas colunas que vocês escolherem, com
`value_counts()` e `describe()`.

⭐ **Pelo menos uma coisa esquisita está lá.** Nas quatro bases do cardápio, está.

Anotem: de onde vieram os dados, quantas linhas e colunas, qual é a coluna-alvo, o
que está faltando (inclusive o que o `.isna()` não vê) e **o que vocês descartaram,
e por quê**.

### 3. Uma reta numa coluna numérica

⚠️ **Leiam com atenção, porque é fácil entender errado.** Na base do banco o alvo de
verdade é sim/não, e a gente **não consegue** modelar isso com uma reta — por isso
hoje a gente previu a idade.

Façam o mesmo: **escolham uma coluna numérica qualquer** da base de vocês e ajustem
uma reta nela, repetindo as Etapas 2 a 7 deste notebook.

- **Plano de saúde** ou **vinho como número**: essa coluna pode ser o alvo de verdade
  do projeto de vocês.
- **Telecom** ou **renda**: não é — e a peça que falta chega amanhã. Isto aqui é
  exercício de dedo, não o modelo do projeto.

Depois: **olhem em que o modelo se apoiou, e perguntem se faz sentido.**